### Create the mart layer for the Skincare Products warehouse

In [ ]:
from google.cloud import bigquery

project_id = "segfault-434120"
dataset = "skincare_products_mrt"
region = "us-central1"

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset segfault-434120.skincare_products_mrt


#### Question 1: What are the most common ingredients in products and what dermatologic diseases do they treat?

##### Data issues
During our LLM standardization of ingredient names across the 3 tables (cosmetic_ingredient, sephora_product, dermatologic_disease), we had some instances where entire batches of ingredients were dropped due to LLM processing errors. This resulted in a somewhat fragmented link between products and dermatologic diseases, which can be seen through the smaller number of of rows in the reporting table.

##### Join `disease_ingredient_treatment` with `product_ingredient` to get the diseases each product treats along with the ingredient used


In [ ]:
%%bigquery
select
  pi.ingredient_unique_name as ingredient_name,
  STRING_AGG(DISTINCT dit.disease_name, ', ') AS treated_diseases,  -- Combine all treated diseases
  count(pi.product_id) as product_count,
from
  skincare_products_int.product_ingredient pi
left join
  skincare_products_int.disease_ingredient_treatment dit
on
  pi.ingredient_unique_name = dit.ingredient_unique_name
where dit.disease_name is not null
group by
  pi.ingredient_unique_name
order by product_count desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient_name,treated_diseases,product_count
0,GLYCOLIC ACID,"Striae, Melasma, Psoriasis, Monilethrix, Chemi...",5111
1,TOCOPHERYL ACETATE,"Striae, Hair Loss, Swimmer's Itch",5010
2,SALICYLIC ACID,"Warts, Hair Loss, Psoriasis, Lyme Disease, Swi...",4020
3,CITRIC ACID,Striae,2209
4,BENZYL ALCOHOL,Head Lice,1326
5,KERATOLYTIC,"Darier Disease, Asteatotic Eczema, Keratosis P...",906
6,SODIUM CHLORIDE,Spider Veins,872
7,ASCORBIC ACID,"Striae, Ehlers-Danlos Syndrome, Progressive Pi...",861
8,PROPYLENE GLYCOL,Pityrosporum Folliculitis,604
9,ALOE BARBADENSIS LEAF EXTRACT,"Hair Loss, Swimmer's Itch",270


##### Add fields from `cosmetic_ingredient` to provide further information about ingredients


In [ ]:
%%bigquery
with common_product_ingredients as (
  select
    pi.ingredient_unique_name as ingredient_name,
    dit.disease_name AS treated_disease,  -- delay combining all treated diseases
    count(distinct pi.product_id) as product_count,
  from
    skincare_products_int.product_ingredient pi
  left join
    skincare_products_int.disease_ingredient_treatment dit
  on
    pi.ingredient_unique_name = dit.ingredient_unique_name
  where dit.disease_name is not null
  group by
    pi.ingredient_unique_name, dit.disease_name
  # order by product_count desc
  # limit 10
),

common_ingredient_treatment_information as (
  select
    cpi.ingredient_name,
    cpi.treated_disease,
    cpi.product_count,
    ci.intended_use,
    ci.regulatory_restriction
  from
    common_product_ingredients cpi
  join
    skincare_products_int.cosmetic_ingredient ci
  on
    cpi.ingredient_name = ci.ingredient_unique_name
)

# Group ingredient treatment information by product
select
  citi.ingredient_name,
  STRING_AGG(DISTINCT citi.treated_disease, ', ') AS treated_diseases,  -- Combine all treated diseases
  max(citi.intended_use) as intended_use,  -- Choose the longest intended_use
  string_agg(DISTINCT citi.regulatory_restriction, ', ') as regulatory_restriction,  -- Concatenate all regulatory restrictions
  sum(citi.product_count) as product_count
from
  common_ingredient_treatment_information citi
group by
  citi.ingredient_name
order by product_count desc
limit 10

# Currently also bugged due to product_ingredient not being correct

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient_name,treated_diseases,intended_use,regulatory_restriction,product_count
0,TOCOPHERYL ACETATE,"Striae, Hair Loss, Swimmer's Itch","ANTIOXIDANT, SKIN CONDITIONING",None,4359
1,SALICYLIC ACID,"Warts, Hair Loss, Psoriasis, Lyme Disease, Swi...","ANTI-SEBORRHEIC, FRAGRANCE, HAIR CONDITIONING,...",III/98\nV/3,3972
2,GLYCOLIC ACID,"Striae, Melasma, Psoriasis, Monilethrix, Chemi...",BUFFERING,None,3780
3,CITRIC ACID,Striae,"BUFFERING, CHELATING, FRAGRANCE",V/59,2021
4,BENZYL ALCOHOL,Head Lice,"PERFUMING, PRESERVATIVE, SOLVENT, VISCOSITY CO...",III/45 and\nV/34,1187
5,KERATOLYTIC,"Darier Disease, Asteatotic Eczema, Keratosis P...",SKIN CONDITIONING,None,876
6,SODIUM CHLORIDE,Spider Veins,"BULKING, FRAGRANCE, ORAL CARE, VISCOSITY CONTR...",None,814
7,ASCORBIC ACID,"Striae, Ehlers-Danlos Syndrome, Progressive Pi...","ANTIOXIDANT, BUFFERING, FRAGRANCE, SKIN CONDIT...",None,789
8,PROPYLENE GLYCOL,Pityrosporum Folliculitis,"HUMECTANT, SKIN CONDITIONING, SOLVENT, VISCOSI...",None,570
9,ALOE BARBADENSIS LEAF EXTRACT,"Hair Loss, Swimmer's Itch","HUMECTANT, ORAL CARE, SKIN CONDITIONING, SKIN ...",None,262


##### Add data for how many brands use each ingredient

In [ ]:
%%bigquery
with common_product_ingredients as (
  select
    pi.ingredient_unique_name as ingredient_name,
    dit.disease_name AS treated_disease,  -- delay combining all treated diseases
    count(distinct pi.product_id) as product_count,
  from
    skincare_products_int.product_ingredient pi
  left join
    skincare_products_int.disease_ingredient_treatment dit
  on
    pi.ingredient_unique_name = dit.ingredient_unique_name
  where dit.disease_name is not null
  group by
    pi.ingredient_unique_name, dit.disease_name
  # order by product_count desc
  # limit 10
),

common_ingredient_treatment_information as (
  select
    cpi.ingredient_name,
    cpi.treated_disease,
    cpi.product_count,
    ci.intended_use,
    ci.regulatory_restriction
  from
    common_product_ingredients cpi
  join
    skincare_products_int.cosmetic_ingredient ci
  on
    cpi.ingredient_name = ci.ingredient_unique_name
),

ingredient_brand_count as (
  select
    pi.ingredient_unique_name as ingredient_name,
    count(distinct b.brand_id) as brand_count
  from
    skincare_products_int.brand b
  left join skincare_products_int.sephora_product sp on sp.brand_id = b.brand_id
  left join skincare_products_int.product_ingredient pi on sp.product_id = pi.product_id
  group by
    pi.ingredient_unique_name
)

# Group ingredient treatment information by product
select
  citi.ingredient_name,
  STRING_AGG(DISTINCT citi.treated_disease, ', ') AS treated_diseases,  -- Combine all treated diseases
  max(citi.intended_use) as intended_use,  -- Choose the longest intended_use
  string_agg(DISTINCT citi.regulatory_restriction, ', ') as regulatory_restriction,  -- Concatenate all regulatory restrictions
  sum(citi.product_count) as product_count,
  sum(ibc.brand_count) as brand_count
from
  common_ingredient_treatment_information citi
left join ingredient_brand_count ibc on ibc.ingredient_name = citi.ingredient_name
group by
  citi.ingredient_name
order by product_count desc
limit 10

# Currently also bugged due to product_ingredient not being correct

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient_name,treated_diseases,intended_use,regulatory_restriction,product_count,brand_count
0,TOCOPHERYL ACETATE,"Striae, Hair Loss, Swimmer's Itch","ANTIOXIDANT, SKIN CONDITIONING",None,4359,495
1,SALICYLIC ACID,"Warts, Hair Loss, Psoriasis, Lyme Disease, Swi...","ANTI-SEBORRHEIC, FRAGRANCE, HAIR CONDITIONING,...",III/98\nV/3,3972,888
2,GLYCOLIC ACID,"Striae, Melasma, Psoriasis, Monilethrix, Chemi...",BUFFERING,None,3780,990
3,CITRIC ACID,Striae,"BUFFERING, CHELATING, FRAGRANCE",V/59,2021,200
4,BENZYL ALCOHOL,Head Lice,"PERFUMING, PRESERVATIVE, SOLVENT, VISCOSITY CO...",III/45 and\nV/34,1187,151
5,KERATOLYTIC,"Darier Disease, Asteatotic Eczema, Keratosis P...",SKIN CONDITIONING,None,876,240
6,SODIUM CHLORIDE,Spider Veins,"BULKING, FRAGRANCE, ORAL CARE, VISCOSITY CONTR...",None,814,153
7,ASCORBIC ACID,"Striae, Ehlers-Danlos Syndrome, Progressive Pi...","ANTIOXIDANT, BUFFERING, FRAGRANCE, SKIN CONDIT...",None,789,189
8,PROPYLENE GLYCOL,Pityrosporum Folliculitis,"HUMECTANT, SKIN CONDITIONING, SOLVENT, VISCOSI...",None,570,103
9,ALOE BARBADENSIS LEAF EXTRACT,"Hair Loss, Swimmer's Itch","HUMECTANT, ORAL CARE, SKIN CONDITIONING, SKIN ...",None,262,118


##### Create the mart table

In [ ]:
%%bigquery
create or replace table skincare_products_mrt.common_ingredient_information as (
  with common_product_ingredients as (
    select
      pi.ingredient_unique_name as ingredient_name,
      dit.disease_name AS treated_disease,  -- delay combining all treated diseases
      count(distinct pi.product_id) as product_count,
    from
      skincare_products_int.product_ingredient pi
    left join
      skincare_products_int.disease_ingredient_treatment dit
    on
      pi.ingredient_unique_name = dit.ingredient_unique_name
    where dit.disease_name is not null
    group by
      pi.ingredient_unique_name, dit.disease_name
    # order by product_count desc
    # limit 10
  ),

  common_ingredient_treatment_information as (
    select
      cpi.ingredient_name,
      cpi.treated_disease,
      cpi.product_count,
      ci.intended_use,
      ci.regulatory_restriction
    from
      common_product_ingredients cpi
    join
      skincare_products_int.cosmetic_ingredient ci
    on
      cpi.ingredient_name = ci.ingredient_unique_name
  ),

  ingredient_brand_count as (
    select
      pi.ingredient_unique_name as ingredient_name,
      count(distinct b.brand_id) as brand_count
    from
      skincare_products_int.brand b
    left join skincare_products_int.sephora_product sp on sp.brand_id = b.brand_id
    left join skincare_products_int.product_ingredient pi on sp.product_id = pi.product_id
    group by
      pi.ingredient_unique_name
  )

  # Group ingredient treatment information by product
  select
    citi.ingredient_name,
    STRING_AGG(DISTINCT citi.treated_disease, ', ') AS treated_diseases,  -- Combine all treated diseases
    max(citi.intended_use) as intended_use,  -- Choose the longest intended_use
    string_agg(DISTINCT citi.regulatory_restriction, ', ') as regulatory_restriction,  -- Concatenate all regulatory restrictions
    sum(citi.product_count) as product_count,
    sum(ibc.brand_count) as brand_count
  from
    common_ingredient_treatment_information citi
  left join ingredient_brand_count ibc on ibc.ingredient_name = citi.ingredient_name
  group by
    citi.ingredient_name
  order by product_count desc
)

Query is running:   0%|          |

""


#### Question 2: Which product categories tend to have the highest average user rating?


##### Add `category` to `sephora_product_review`


In [ ]:
%%bigquery
select
  c.category_name,
  spr.*
from
  skincare_products_int.sephora_product_review spr
left join skincare_products_int.product_category pc on pc.product_id = spr.product_id
left join skincare_products_int.category c on c.category_id = pc.category_id

Query is running:   0%|          |

Downloading:   0%|          |

,category_name,review_id,author_id,rating,is_recommended,helpfulness,review_feedback_count,review_upvote_count,review_downvote_count,submission_time,...,skin_tone,new_eye_color,skin_type,hair_color,product_id,product_name,price_usd,_data_source,_load_time,brand_id
0,Face Serums,659942,5423143692,5,True,0.863636,44,38,6,2019-03-06,...,lightMedium,blue,normal,brown,P427420,Multi-Peptide + HA Serum,30.8,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6234
1,Treatments,659942,5423143692,5,True,0.863636,44,38,6,2019-03-06,...,lightMedium,blue,normal,brown,P427420,Multi-Peptide + HA Serum,30.8,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6234
2,Eye Creams & Treatments,662013,6653453276,4,True,0.125000,16,2,14,2019-07-17,...,fairLight,hazel,oily,blonde,P447210,Squalane + Marine Algae Firming & Lifting Eye ...,56.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6195
3,Face Masks,383147,1911184186,1,False,0.684211,19,13,6,2021-09-12,...,lightMedium,green,combination,brown,P429952,Jet Lag Mask,49.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6247
4,Skincare,332990,11211006059,2,False,0.285714,7,2,5,2021-06-07,...,fair,brown,normal,brown,P442840,Barrier+ Triple Lipid-Peptide Face Cream,54.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3121972,Skincare,463760,10183487671,5,True,0.333333,3,1,2,2020-10-26,...,lightMedium,brown,dry,auburn,P442840,Barrier+ Triple Lipid-Peptide Face Cream,54.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6289
3121973,Moisturizers,175433,2040907133,5,True,0.333333,3,1,2,2020-07-28,...,tan,brown,oily,black,P12045,Grape Water Moisturizing Face Mist,12.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,4171
3121974,Moisturizers,1080547,22565329233,3,False,0.333333,3,1,2,2022-08-08,...,light,hazel,combination,brown,P471237,Rose & Hyaluronic Acid Deep Hydration Moisturizer,46.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,4348
3121975,Mists & Essences,142244,2555683473,5,<NA>,0.333333,3,1,2,2015-09-21,...,None,None,None,None,P399623,Luminous Dewy Skin Mist,49.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00,6041


##### Reduce review_category to category_name and average rating

In [ ]:
%%bigquery
with review_category as (
  select
    c.category_name,
    spr.*
  from
    skincare_products_int.sephora_product_review spr
  left join skincare_products_int.product_category pc on pc.product_id = spr.product_id
  left join skincare_products_int.category c on c.category_id = pc.category_id
)

select
  category_name,
  CAST(avg(rating) as FLOAT64) as average_rating,
  count(review_id) as review_count
from
  review_category
group by
  category_name
order by
  review_count desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,category_name,average_rating,review_count
0,Skincare,4.299158,1094411
1,Moisturizers,4.320259,503523
2,Treatments,4.304316,222042
3,Cleansers,4.344181,200604
4,Face Serums,4.286048,174600
5,Face Wash & Cleansers,4.300258,121722
6,Mini Size,4.285621,85498
7,Eye Care,4.178536,74999
8,Masks,4.341127,70531
9,Eye Creams & Treatments,4.182112,70440


##### Create the mart table

In [ ]:
%%bigquery
create or replace table skincare_products_mrt.category_rating as (
  with review_category as (
    select
      c.category_name,
      spr.*
    from
      skincare_products_int.sephora_product_review spr
    left join skincare_products_int.product_category pc on pc.product_id = spr.product_id
    left join skincare_products_int.category c on c.category_id = pc.category_id
  )

  select
    category_name,
    CAST(avg(rating) as FLOAT64) as average_rating,
    count(review_id) as review_count
  from
    review_category
  group by
    category_name
  order by
    review_count desc
)

Query is running:   0%|          |

""


#### **Question 3: Which skincare products are highly rated by users with specific skin types (e.g., dry, oily, combination)?**

##### Create a table for highly rated products for dry skin type

In [ ]:
%%bigquery
create or replace table skincare_products_mrt.highly_rated_dry_skin as
select
    p.product_name,
    avg(r.rating) as average_rating,
    sum(pr.review_count) as total_reviews
from
    skincare_products_int.sephora_product_review r
join
    skincare_products_int.sephora_product p
    on r.product_id = p.product_id
join
    (select
        product_id,
        count(*) as review_count
     from
        skincare_products_int.sephora_product_review
     group by
        product_id) pr
    on p.product_id = pr.product_id
where
    r.skin_type = 'dry'
group by
    p.product_name
order by
    average_rating desc

Query is running:   0%|          |

""


##### Create a table for highly rated products for oily skin type

In [ ]:
%%bigquery
create or replace table skincare_products_mrt.highly_rated_oily_skin as
select
    p.product_name,
    avg(r.rating) as average_rating,
    sum(pr.review_count) as total_reviews
from
    skincare_products_int.sephora_product_review r
join
    skincare_products_int.sephora_product p
    on r.product_id = p.product_id
join
    (select
        product_id,
        count(*) as review_count
     from
        skincare_products_int.sephora_product_review
     group by
        product_id) pr
    on p.product_id = pr.product_id
where
    r.skin_type = 'oily'
group by
    p.product_name
order by
    average_rating desc

Query is running:   0%|          |

""


##### Create a table for highly rated products for combination skin type


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.highly_rated_combination_skin as
select
    p.product_name,
    avg(r.rating) as average_rating,
    sum(pr.review_count) as total_reviews
from
    skincare_products_int.sephora_product_review r
join
    skincare_products_int.sephora_product p
    on r.product_id = p.product_id
join
    (select
        product_id,
        count(*) as review_count
     from
        skincare_products_int.sephora_product_review
     group by
        product_id) pr
    on p.product_id = pr.product_id
where
    r.skin_type = 'combination'
group by
    p.product_name
order by
    average_rating desc

Query is running:   0%|          |

""


##### Create a table for highly rated products for sensitive skin type


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.highly_rated_sensitive_skin as
select
    p.product_name,
    avg(r.rating) as average_rating,
    sum(pr.review_count) as total_reviews
from
    skincare_products_int.sephora_product_review r
join
    skincare_products_int.sephora_product p
    on r.product_id = p.product_id
join
    (select
        product_id,
        count(*) as review_count
     from
        skincare_products_int.sephora_product_review
     group by
        product_id) pr
    on p.product_id = pr.product_id
where
    r.skin_type = 'sensitive'
group by
    p.product_name
order by
    average_rating desc

Query is running:   0%|          |

""


##### Create an aggregate table for average ratings by skin condition types, excluding nulls

In [ ]:
%%bigquery
create or replace table skincare_products_mrt.aggregate_skin_type_ratings as
select
    sr.skin_type as skin_condition,
    avg(sr.rating) as average_rating
from
    skincare_products_int.sephora_product_review sr
where
    sr.skin_type is not null
group by
    skin_condition
order by
    skin_condition

Query is running:   0%|          |

""


#### **Question 4: What is the average price of products that contain certain popular ingredients?**

##### Create a table for average price and product count of popular ingredients


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.average_price_by_ingredient as
select
    i.ingredient_unique_name,
    avg(pp.price_usd) as average_price,
    count(distinct p.product_id) as product_count
from
    skincare_products_int.product_ingredient i
join
    skincare_products_int.sephora_product p
on
    i.product_id = p.product_id
join
    skincare_products_int.product_price pp
on
    p.product_id = pp.product_id
group by
    i.ingredient_unique_name
order by
    product_count desc

Query is running:   0%|          |

""


#### **Question 5: Are there any pricing trends (e.g., discounts, sales) related to products that are out of stock or online exclusives?**

##### Create table for pricing trends related to limited edition products


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.limited_edition_pricing_trends as
select
    p.product_name,
    pr.price_usd,
    pr.sale_price_usd,
    pv.variation_min_price,
    pv.variation_max_price,
    pv.variation_count
from
    skincare_products_int.product_stock_status ps
join
    skincare_products_int.product_price pr
    on ps.product_id = pr.product_id
join
    skincare_products_int.product_variation pv
    on ps.product_id = pv.product_id
join
    skincare_products_int.sephora_product p
    on ps.product_id = p.product_id
where
    ps.limited_edition = true

Query is running:   0%|          |

""


##### Create table for pricing trends related to new products


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.new_product_pricing_trends as
select
    p.product_name,
    pr.price_usd,
    pr.sale_price_usd,
    pv.variation_min_price,
    pv.variation_max_price,
    pv.variation_count
from
    skincare_products_int.product_stock_status ps
join
    skincare_products_int.product_price pr
    on ps.product_id = pr.product_id
join
    skincare_products_int.product_variation pv
    on ps.product_id = pv.product_id
join
    skincare_products_int.sephora_product p
    on ps.product_id = p.product_id
where
    ps.new_product = true

Query is running:   0%|          |

""


##### Create table for pricing trends related to online-only products


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.online_only_pricing_trends as
select
    p.product_name,
    pr.price_usd,
    pr.sale_price_usd,
    pv.variation_min_price,
    pv.variation_max_price,
    pv.variation_count
from
    skincare_products_int.product_stock_status ps
join
    skincare_products_int.product_price pr
    on ps.product_id = pr.product_id
join
    skincare_products_int.product_variation pv
    on ps.product_id = pv.product_id
join
    skincare_products_int.sephora_product p
    on ps.product_id = p.product_id
where
    ps.online_only = true

Query is running:   0%|          |

""


##### Create table for pricing trends related to out-of-stock products


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.out_of_stock_pricing_trends as
select
    p.product_name,
    pr.price_usd,
    pr.sale_price_usd,
    pv.variation_min_price,
    pv.variation_max_price,
    pv.variation_count
from
    skincare_products_int.product_stock_status ps
join
    skincare_products_int.product_price pr
    on ps.product_id = pr.product_id
join
    skincare_products_int.product_variation pv
    on ps.product_id = pv.product_id
join
    skincare_products_int.sephora_product p
    on ps.product_id = p.product_id
where
    ps.out_of_stock = true

Query is running:   0%|          |

""


##### Create table for pricing trends related to sephora-exclusive products


In [ ]:
%%bigquery
create or replace table skincare_products_mrt.sephora_exclusive_pricing_trends as
select
    p.product_name,
    pr.price_usd,
    pr.sale_price_usd,
    pv.variation_min_price,
    pv.variation_max_price,
    pv.variation_count
from
    skincare_products_int.product_stock_status ps
join
    skincare_products_int.product_price pr
    on ps.product_id = pr.product_id
join
    skincare_products_int.product_variation pv
    on ps.product_id = pv.product_id
join
    skincare_products_int.sephora_product p
    on ps.product_id = p.product_id
where
    ps.sephora_exclusive = true

Query is running:   0%|          |

""


##### Create an aggregate table for average prices by stock status

In [ ]:
%%bigquery
create or replace table skincare_products_mrt.aggregate_stock_status_prices as
select
    case
        when ps.limited_edition then 'limited_edition'
        when ps.new_product then 'new_product'
        when ps.online_only then 'online_only'
        when ps.out_of_stock then 'out_of_stock'
        when ps.sephora_exclusive then 'sephora_exclusive'
    end as stock_status,
    avg(pp.price_usd) as average_price
from
    skincare_products_int.product_stock_status ps
join
    skincare_products_int.product_price pp
on
    ps.product_id = pp.product_id
where
    ps.limited_edition or
    ps.new_product or
    ps.online_only or
    ps.out_of_stock or
    ps.sephora_exclusive
group by
    stock_status
order by
    stock_status

Query is running:   0%|          |

""
